# Tamaños de orden en la liquidez media (3.2.4, diseño en estrella, desviación 12): BF

**Qué hace:** corre el sistema final, TWAP y VWAP en la liquidez **media** de la v3 con las órdenes multiplicadas por `FACTOR_TAMANO` (celda 10):
- **0,5** → 5 % del volumen del día (2,5 % en el cierre);
- **1,5** → 15 % del volumen del día (7,5 % en el cierre).

Es todo igual a `Colab_90corridas_media_A/B` de PS (mismo código, rama `feature/ps-3-2-1-cuadernos`; mismas órdenes, semillas 4, 5 y 6, impacto acumulado, corrección del cierre y sistema final). **Solo cambia el tamaño**, así que cada corrida queda pareada con la media ×1 de PS.

**Solo se corre si el equipo aprobó el diseño en estrella** (Sprint Review del 6 de noviembre).

**Cómo usarlo:**
1. Abre este cuaderno **dos veces** (dos pestañas). En la celda 10 deja `PARTE = 'A'` en una y `PARTE = 'B'` en la otra, con el **mismo** `FACTOR_TAMANO`.
2. En cada una, «Entorno de ejecución → Ejecutar todas». Cada parte dura ≈ **3 horas**. Si Colab se corta, vuelve a «Ejecutar todas»: retoma donde quedó.
3. La celda 13 debe decir **LISTO**. Si TWAP o VWAP quedan bajo el 95 % en alguna orden de cierre, lo informa aparte: no se repite (la semilla fija el día) y se analiza con el costo ajustado.
4. La parte que termina última arma la tabla (celda 14) y el zip `tamanos_media_f<FACTOR>.zip` (celda 15).

**Necesita:** los checkpoints del sistema final en Drive (`slippage_colab/resultados/sistema_final/maestro/`), los mismos de las 90 corridas. Si no los tienes, pídele la carpeta a Paolo.

**Límite que se declara:** el Maestro se entrenó con un solo tamaño (10 % y 5 % del día); 0,5 y 1,5 quedan fuera de lo que vio.


## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-3-2-1-cuadernos, o main si ya se mergeó; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && (git checkout -q -f origin/feature/ps-3-2-1-cuadernos || git checkout -q -f origin/main)
!cd /content/slippage && git log --oneline -1 && grep -c "semilla_inicial" scripts/colab/evaluar_sistema_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Parámetros

In [ ]:
# CELDA 10: PARÁMETROS (cambia FACTOR_TAMANO y PARTE; lo demás es igual a las 90 corridas)
FACTOR_TAMANO = 0.5         # 0.5 (5 % del día) o 1.5 (15 % del día); en el cierre, la mitad
PARTE = 'A'                 # A = MO-01 a MO-05, B = MO-06 a MO-10
ESCENARIO = 'media'         # fijo: el eje de tamaño se mide en la liquidez media (diseño en estrella)
ORDENES = {'A': 'MO-01 MO-02 MO-03 MO-04 MO-05', 'B': 'MO-06 MO-07 MO-08 MO-09 MO-10'}[PARTE]
SEMILLA_INICIAL = 4         # 4 = semillas 4, 5 y 6: las mismas de las 90 corridas, para parear con la media x1
SEMILLAS = 3
assert FACTOR_TAMANO in (0.5, 1.5), 'FACTOR_TAMANO debe ser 0.5 o 1.5'
assert PARTE in ('A', 'B')
FACTOR = f'{FACTOR_TAMANO:g}'
OUT = f'{RESULTS_DIR}/tamanos/{ESCENARIO}_f{FACTOR}/{PARTE}'
print('Factor de tamaño', FACTOR, '| escenario', ESCENARIO, '| parte', PARTE, '| órdenes', ORDENES,
      '| semillas', list(range(SEMILLA_INICIAL, SEMILLA_INICIAL + SEMILLAS)), '| carpeta', OUT)

## Corridas

In [ ]:
# CELDA 11: sistema final con el factor de tamaño (15 corridas, ≈ 1 h). Si se corta, vuelve a ejecutar y retoma donde quedó
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes {ORDENES} --seeds {SEMILLAS} --semilla-inicial {SEMILLA_INICIAL} --escenario {ESCENARIO} --factor-tamano {FACTOR_TAMANO} --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --out-dir {OUT} --max-minutes 330 --completar-al-cierre --modo muestreo --ckpt-maestro {RESULTS_DIR}/sistema_final/maestro/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/sistema_final/maestro/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/sistema_final/maestro/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/sistema_final/maestro/ejecutor_cierre.pt --nombre PPO_sistema 2>&1 | grep -v UserWarning

In [ ]:
# CELDA 12: TWAP y VWAP con las mismas órdenes, semillas y factor de tamaño (30 corridas, ≈ 2 h)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/benchmarks_meta_ordenes_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes {ORDENES} --seeds {SEMILLAS} --semilla-inicial {SEMILLA_INICIAL} --escenario {ESCENARIO} --factor-tamano {FACTOR_TAMANO} --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --out-dir {OUT} --max-minutes 330 2>&1 | grep -v UserWarning

## Revisión

In [ ]:
# CELDA 13: revisión (15 del sistema, 15 de TWAP y 15 de VWAP, todas con el factor de tamaño y las semillas 4 a 6)
import json, glob
esperado = {'PPO_sistema': 15, 'TWAP': 15, 'VWAP': 15}
conteo, errores, sistema_bajo, bench_bajo, otras = {}, [], [], [], []
for f in glob.glob(f'{OUT}/*.json'):
    for c in json.load(open(f, encoding='utf-8')).get('corridas', {}).values():
        if 'error' in c:
            errores.append((c['orden'], c['estrategia'], c['seed'], c['error'][:80]))
            continue
        conteo[c['estrategia']] = conteo.get(c['estrategia'], 0) + 1
        if c.get('factor_tamano') != FACTOR_TAMANO or c['seed'] not in (4, 5, 6):
            otras.append((c['orden'], c['estrategia'], c['seed'], c.get('factor_tamano')))
        if c.get('execution_rate', 1.0) < 0.95:
            (sistema_bajo if c['estrategia'] == 'PPO_sistema' else bench_bajo).append(
                (c['orden'], c['estrategia'], c['seed'], round(c['execution_rate'], 3)))
print('Corridas:', conteo, '(se esperan', esperado, ')')
print('Errores:', errores or 'ninguno')
print('Factor o semilla distintos de lo esperado:', otras or 'ninguno')
print('Sistema con cumplimiento < 0,95:', sistema_bajo or 'ninguno')
print('TWAP/VWAP con cumplimiento < 0,95 (no se repiten; se analizan con el costo ajustado):', bench_bajo or 'ninguno')
if conteo == esperado and not errores and not otras and not sistema_bajo:
    print('LISTO')
elif conteo != esperado or errores:
    print('FALTA ALGO: vuelve a ejecutar las celdas 11 y 12 (retoman donde quedaron)')
else:
    print('REVISAR: copia esta salida completa para Claude Code (repetir no cambia el resultado: la semilla fija el día)')

## Tabla del factor (cuando A y B terminaron)

In [ ]:
# CELDA 14: tabla del factor de tamaño (solo cuando A y B del mismo factor terminaron)
import os
base = f'{RESULTS_DIR}/tamanos/{ESCENARIO}_f{FACTOR}'
listos = [p for p in ('A', 'B') if len(glob.glob(f'{base}/{p}/*.json')) == 2]
print('Partes con sus 2 archivos:', listos)
if len(listos) == 2:
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/consolidar_resultados_3_2_2.py --entradas {base}/A {base}/B --sistema PPO_sistema --out-dir {base}/tabla")
    print(open(f'{base}/tabla/tabla_maestra.md', encoding='utf-8').read())
else:
    print('Falta la otra parte: cuando termine, la tabla y el zip salen de esa.')

## Zip para mandar

In [ ]:
# CELDA 15: zip del factor para mandarlo (sale completo de la parte que termina última)
import zipfile
base = f'{RESULTS_DIR}/tamanos/{ESCENARIO}_f{FACTOR}'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) for a in fs if a.endswith(('.json', '.csv', '.md', '.png'))]
zpath = f'{RESULTS_DIR}/tamanos_{ESCENARIO}_f{FACTOR}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath, '(descárgalo desde slippage_colab/resultados en Drive)')